<h1>Reading and writing .npz files</h1>

This is a brief overview of how `.npz` files are used to store various adjacency matrices, and their associated states. It is my summary of interactions with my AI overlord.

The impetus for this was the need to save my data on the "orbit graph" -- the graph showing how the 2-2 move affects the non-isomorphic parity face vectors on graphs (here, "non-isomorphic" refers to using *only* the graph symmetries, not those of the combined state). For $|V| = 16, there are 183,191 states, so some efficiency is necessary to avoid large files.

<h2>Reference</h2>

Before getting into the mechanics of how these files are organized, here we put the keyword for the important files, the graph edge lists `V={numVert}-edgelist.npz`, the Pachner adjacency matrices `V={numVert}-Pachner-graph-{typeStr}.npz`, and the two 2-2 move face matrices `V={numVert}-single-face-move-matrix.npz` and `V={numVert}-noniso-face-vec-matrix.npz`.

* `V={numVert}-edgelist.npz`
    + `edge_lists`: A `(numGraph, |E|, 2)` `numpy` array, where `numGraph` is the number of graphs for the chosen vertex number, with each entry along `axis = 0` is a non-isomorphic `(|E|, 2)` array of the edges `[start, end]` of the graph.

* `V={numVert}-Pachner-graph-{typeStr}.npz`
    + `graphs`: A `(numGraph,)` `numpy` array of the canonical graph labels (from the edge list given in the last file)
    + `orientations`: If present in file, for `graphs[iii]`, the value `orientations[iii]` is a bitmask, whose binary representation gives the edge orientations for the graph. This number is bounded by $2^{|E|}$, and the digit at place $jjj$ (i.e. the coefficient of $2^j$) in the bitmask is 1 if the orientation points with the canonical orientation $start \to end$, and 0 otherwise.
    + `parities`: If present in file, for `graphs[iii]`, the value `parities[iii]` is a bitmask, whose binary representation gives the edge parities for the graph. This number is bounded by $2^{|E|}$, and the digit at place $jjj$ (i.e. the coefficient of $2^j$) in the bitmask is 1 if the edge parity is odd, and 0 if it is even.
    + `matrix_data`, `matrix_indices`, `matrix_indptr`, `matrix_shape`: Standard arrays for the CSR matrix representing the adjacency matrix of the Pachner graph.

* `V={numVert}-single-face-move-matrix.npz`
    + `metadata`: A `(numGraph, numFaces)` size `numpy` array, giving the graph and face label for each state.
    + `sparse_data`, `sparse_indices`, `sparse_indptr`, `spares_shape`: Standard arrays for the CSR matrix representing the action of the 2-2 move on all graph faces. I know, I should have made the names consistent with the Pachner graph.

* `V={numVert}-noniso-face-vec-matrix.npz`
    + `graphs`: A `numpy` array of the canonical graph labels (from the edge list given in the last file). The number of entries for this, and the next, array is the number of non-isomorphic face vectors for the given choice of $|V|$, which has no easy formula.
    + `face_vec`: A `numpy` array of the non-isomorphic face vectors, in combination with `graphs`, so `face_vec[iii]` is given in terms of the standard labels of the faces for `graphs[iii]` (as determined by `trivalent`, using the canonical edge list for the graph).
    + `sparse_data`, `sparse_indices`, `sparse_indptr`, `spares_shape`: Standard arrays for the CSR matrix representing the action of the 2-2 move on the non-isomorphic face vectors.

<h2>Saving state data</h2>

First, we want to save the matrix related to how the 2-2 move acts on these states; note that we use the single-move matrices for both individual faces and non-isomorphic face vectors, rather than the corresponding orbit graph distance matrices, since the latter will not be sparse matrices. On the other hand, each 2-2 move maps a face to only one other, so that reduces the number of entries in the single-move graph to only one per `numFaces` in each row, for single faces. Thus, we save the matrix as a sparse CSR matrix. Since each entry will be relative small, we store it as a `np.uint8` (unsigned 8-bit integer) variable, so only 1 byte per element.

It turns out that we can store the state information as well in the same file as metadata. For the non-isomorphic face vectors, the information is given as `(graphNum, faceVec)`, where `graphNum` is the numerical graph label in the canonical list, and `faceVec` is a list of face labels. If we store `graphNum` as a `np.uint16` variable (unsigned 16-bit integer), then it can be a number up to $2^{16} = 65,536$, which means it is fine up to $|V| = 22$. For $|V| = 24$, there are 97,033 planar 3-connected graphs. By duality, the number of entries in `faceVec` will be less than or equal to `numFace // 2`. and each entry is of maximum size `numFace`. Each face label fits easily into a `np.uint8` variable.

Here is the example and table presented by my AI overlord. Suppose we have $|V| = 18$, with 2274 graphs, and `numFace` is 11. Then with the state `(2273, (1, 3, 5, 7, 9, 11))`, we have

| format              | storage for example state                                                                         |
|:-------------------:|:-------------------------------------------------------------------------------------------------:|
| text                | '2274' (4 bytes) + ' ' (1 byte) + (face labels + spaces) (6 * 2 bytes) + '\n' (1 byte) = 18 bytes |
| uncompressed binary | '2274' (2 byte) + (face labels) (6 bytes) = 8 bytes                                               |
| compressed binary   | Entropy coding compresses the repeating padding and structural patterns = ~1 to 2 bytes           |

To implement this compressed binary version of the graph + parity states, here is the following code. Note it is missing `states_list`, so don't run this!

In [ ]:
import numpy as np

# Simulation of your data generation for |V| = 18
# states_list = [(graphNum, [face1, face2, ...]), ...]
MAX_FACE_LEN = 6  # 1 + 18 // 4 (adjust if needed)
SENTINEL = 255    # Indicates an empty slot

graph_ids = []
face_matrices = []

for graph_num, face_vec in states_list:
    graph_ids.append(graph_num)
    
    # Pad the face vector with 255 up to the maximum length
    padded_vec = face_vec + [SENTINEL] * (MAX_FACE_LEN - len(face_vec))
    face_matrices.append(padded_vec)

# Convert to optimized bit-width NumPy arrays`
graph_arr = np.array(graph_ids, dtype=np.uint16)
face_arr = np.array(face_matrices, dtype=np.uint8)

# Save to a single, highly-compressed binary file
np.savez_compressed('quotient_states_V18.npz', graphs=graph_arr, faces=face_arr)

The variable `MAX_FACE_LEN` sets the largest possible value of the face labels. The list of face labels is then padded with $2^8$ if there is no face in that slot, up to the largest possible length of a face vector. The `numpy` arrays are then compressed by Gzip.

The state information is stored in two arrays, since the face vectors have variable length. Thus, `graph_arr` is an 1D array of size `(numStates, )` while `face_arr` is a 2D array of size `(numStates, numFaces)`.

To load the information back in, we have the code below.

In [ ]:
import numpy as np

# Load the compressed file
data = np.load('quotient_states_V18.npz')
graph_arr = data['graphs']
face_arr = data['faces']

# If you need to reconstruct a specific state on the fly:
# graph_num = graph_arr[i]
# face_vec = [f for f in face_arr[i] if f != 255]

Note that the keywords `graphs` and `faces` allow these two arrays to easily be pulled into new arrays for subsequent use. As the last commented text says, it is straightfoward to pull out the information for state `i` from the data.

The same thing can be done with the single-face state data. For this, the states are given by `(graphNum, faceLabel)`, where `faceLabel` is a single number less than or equal to `numFace`. Thus, each state is always represented by two numbers, making the codifying even simpler. Rather than using two separate arrays as `graph_arr` and `face_arr` above, we can put all the information into a single array.

In [ ]:
# rows align 1:1 with your single-face CSR matrix indices
# Column 0: graphNum (uint16 handles up to 65,535)
# Column 1: faceNum  (uint8 handles up to 255)
single_face_metadata = np.array([
    [0, 0],  # Matrix Index 0: Graph 0, Face 0
    [0, 1],  # Matrix Index 1: Graph 0, Face 1
    # ...
    [2273, 10] # Matrix Index 25013: Graph 2273, Face 10
], dtype=np.uint16)

<h2>Saving move matrix with state data</h2>

It turns out, though, that we can use *one* file to store both the state data, along with the matrix representing the 2-2 move acting on these states (either single faces or non-isomorphic face vectors). Here is how to do it for the non-isomorphic face vector case; the single-face situation is similar.

Suppose we have a sparse CSR matrix `adj_matrix`, representing the action of the 2-2 move on the face vectors, whose graph labels and face vectors are `graph_arr` and `face_arr`, respectively. Then we can store this latter metadata along with the CSR matrix as shown below. This can be done because the CSR matrix is represented by three `numpy` arrays `data`, `indices`, `indptr`, and a tuple `shape`. Passing along additional arrays to the Gzip algorithm for compression is not much extra. This means that the state index `i` lines up with both the entry in the matrix `adj_matrix` and the state information.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# --- 1. Your Existing Setup ---
# Let's assume you've built your graph_arr and face_arr (from the previous step)
# And you have your sparse adjacency matrix: `adj_matrix` (a scipy.sparse.csr_matrix)
# Both have a length/dimension of N = 183,191

# --- 2. Save Everything Together ---
np.savez_compressed(
    'quotient_universe_V16.npz',
    # State Metadata
    graphs=graph_arr,
    faces=face_arr,
    # Sparse Matrix Internal Arrays
    sparse_data=adj_matrix.data,
    sparse_indices=adj_matrix.indices,
    sparse_indptr=adj_matrix.indptr,
    sparse_shape=adj_matrix.shape
)

To pull the information back out, we use the labels for all of the arrays as appropriate.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# Load the single unified file
archive = np.load('quotient_universe_V16.npz')

# 1. Reconstruct the exact CSR matrix
adj_matrix = csr_matrix(
    (archive['sparse_data'], archive['sparse_indices'], archive['sparse_indptr']),
    shape=archive['sparse_shape']
)

# 2. Grab the metadata arrays
graphs = archive['graphs']
faces = archive['faces']

print(f"Successfully loaded a state space of size: {adj_matrix.shape[0]}")

Then looking for a specific state, or finding the neighbors of that state, is a few lines of code.

In [ ]:
# Say you are performing a Breadth-First Search (BFS) or short-path query
# and the algorithm tells you that you've arrived at row index: 42069

# Find out exactly what physical state row 42069 represents:
graph_id = graphs[42069]
raw_face_vec = faces[42069]
face_vec = [f for f in raw_face_vec if f != 255] # Strip the padding

print(f"Node 42069 is Graph #{graph_id} with Parity Face Vector: {face_vec}")


# Conversely, if you want to find who Node 42069 connects to in 1 move:
# Because it's a CSR matrix, pulling a row's neighbors is incredibly fast
start_ptr = adj_matrix.indptr[42069]
end_ptr = adj_matrix.indptr[42069 + 1]
neighbor_indices = adj_matrix.indices[start_ptr:end_ptr]

print(f"Node 42069 connects directly to matrix rows: {neighbor_indices}")

<h2>Pachner graphs</h2>

The above used the 2-2 move orbit graphs as examples, but this can be done for the Pachner graphs as well. Some specific implementation details are as follows.

1. The orientation and parity bitmasks will be binary numbers of size $2^{|E|}$. Thus, as long as the number of edges is $<32$, we can use `np.uint32` integers for them. This gives a limit of $|V| = 20$ with $|E| = 30$, since for $|V| = 22$, $|E| = 33$.
2. As before, using `np.uint16` for the graph labels is sufficient up to $|V| = 22$.

The calculation of the number of compression is similar to that given above. Here is the corresponding code for the Pachner graphs.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# --- 1. Artificial setup representing your parsed data ---
# Let's say your full Pachner sector has N states
graph_list = [2274, 2274, 2275]
orient_list = [134217727, 0, 8388607]  # Integer representations of bitmasks
parity_list = [67108863, 1024, 4095]

# Convert configurations to optimized numpy arrays
graphs_arr = np.array(graph_list, dtype=np.uint16)
orient_arr = np.array(orient_list, dtype=np.uint32)
parity_arr = np.array(parity_list, dtype=np.uint32)

# Assume `pachner_adj` is the scipy.sparse.csr_matrix for this sector
# Its rows/columns match the indices of the arrays above 1-to-1

# --- 2. Bundle Everything into One Compressed File ---
np.savez_compressed(
    'pachner_parity_orientation_V18.npz',
    # State Configurations
    graphs=graphs_arr,
    orientations=orient_arr,
    parities=parity_arr,
    # Pachner Matrix Components
    matrix_data=pachner_adj.data,
    matrix_indices=pachner_adj.indices,
    matrix_indptr=pachner_adj.indptr,
    matrix_shape=pachner_adj.shape
)

According to my AI overlord:

>If you ever need to inspect the actual binary "edge colors" for a calculation, you can use Python's fast bitwise operations or `np.unpackbits` rather than string parsing

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# Load the bundle
archive = np.load('pachner_parity_orientation_V18.npz')

# Reconstruct the Pachner graph
pachner_graph = csr_matrix(
    (archive['matrix_data'], archive['matrix_indices'], archive['matrix_indptr']),
    shape=archive['matrix_shape']
)

# Grab the aligned configurations
graphs = archive['graphs']
orientations = archive['orientations']
parities = archive['parities']

# Example: Inspect state at matrix row index i
i = 0
graph_id = graphs[i]
orient_mask = orientations[i]

# If you need to see if edge #5 is flipped:
edge_5_flipped = bool((orient_mask >> 5) & 1)

# Or print it as a clean 27-digit binary string for debugging:
num_edges = 27
binary_string = f"{orient_mask:0{num_edges}b}"